# Samaritan solver on an A100 — Qwen3.8-27B (turnkey, self-verifying)

Serves **Qwen3.8-27B** (Q8_0 MTP GGUF) via **Ollama** on this Colab A100 and
exposes it through a cloudflared tunnel. Every cell checks its own work, and
**cell 4 proves the public URL answers end-to-end before it prints anything**
— so a broken chain fails *here*, in Colab, not later on your laptop.

**Runtime → Change runtime type → A100 GPU, then Runtime → Run all.**

**If Colab disconnected and you're back:** the runtime wiped Ollama, so a stale
tunnel just returns 530. Don't hunt for a URL — **Run all again** (or run the
**Health check** cell to see what's down). Your local eval's `RESUME` file means
you only redo unfinished items.

**Honest limits:** Colab isn't 24/7 and reclaims A100s without warning; Ollama
has no auth, so the random tunnel URL is the only guard — don't share it. Stop
it yourself when done (Runtime → Disconnect and delete runtime); there is no
shutdown cell, on purpose — in a Run-all it would kill the tunnel you just made.

## 1. Confirm the A100

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 2. Install Ollama and start the daemon

Installs `zstd` (Ollama's installer needs it; Colab lacks it), wipes any partial
install, and **asserts the `llama-server` runner landed** — a partial extract
serves `/v1` but 500s at generation. Then starts the daemon (Colab has no
systemd) and waits until it answers. Safe to re-run after a reconnect.

The daemon is started with **flash attention on and an 8-bit (`q8_0`) KV
cache** — together they roughly halve the context cache, which is what lets a
128k window (cell 3) fit alongside the ~30 GB model on one 40 GB A100.

In [ ]:
import os, subprocess, time, urllib.request, glob, shutil
!apt-get -qq install -y zstd || (apt-get -qq update && apt-get -qq install -y zstd)
!rm -rf /usr/local/lib/ollama
!curl -fsSL https://ollama.com/install.sh | sh
ob = shutil.which('ollama')
runner = glob.glob('/usr/local/lib/ollama/**/llama-server', recursive=True)
assert ob and runner, f'install incomplete: ollama={ob} runner={runner} (see output above)'
# (re)start the daemon — kill any old one first so a reconnect starts clean.
try:
    srv.terminate()
except Exception:
    pass
subprocess.run(['pkill', '-f', 'ollama serve'], check=False)
time.sleep(2)
# Flash attention + q8_0 KV cache halve the context cache; needed for 128k
# ctx (cell 3) to fit next to the ~30 GB model. KV quant only bites with FA on.
env = {**os.environ, 'OLLAMA_HOST': '127.0.0.1:11434', 'OLLAMA_KEEP_ALIVE': '-1',
       'OLLAMA_FLASH_ATTENTION': '1', 'OLLAMA_KV_CACHE_TYPE': 'q8_0'}
srv = subprocess.Popen(['ollama', 'serve'], stdout=open('ollama.log', 'w'),
                       stderr=subprocess.STDOUT, env=env)
up = False
for _ in range(30):
    try:
        if urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=3).status == 200:
            up = True; break
    except Exception:
        time.sleep(2)
assert up, 'ollama daemon did not come up:\n' + open('ollama.log').read()[-800:]
print('[OK] ollama daemon UP; runner at', runner[0])

## 3. Pull the model, alias it, and warm it

`qwen3.8:27b-mtp-q8_0` (~30 GB, MTP draft head for faster gen) → aliased to
`samaritan-playout` (the name the harness asks for) with a **128k context**
(`num_ctx 131072`). This is the fix that matters: past `num_ctx` Ollama does
*not* error — it silently drops the oldest tokens, so a long derivation forgets
its own early work and loops, which is what produced the answer-less truncations
at the old 16k. The hybrid attention (only 16 of 64 layers keep a growing KV
cache) plus the 8-bit cache from cell 2 hold 128k to ~4 GB, a comfortable fit.
Then a local generation call **loads it into VRAM and proves it answers** — so
the ~30 GB load happens here, not on your first real request. First run: several
minutes for the pull + load.

In [ ]:
import subprocess, urllib.request, json
TAG = 'qwen3.8:27b-mtp-q8_0'   # plain fallback: 'qwen3.8:27b-q8_0'
subprocess.run(['ollama', 'pull', TAG], check=True)
with open('Modelfile', 'w') as f:
    f.write(f'FROM {TAG}\nPARAMETER num_ctx 131072\n')
subprocess.run(['ollama', 'create', 'samaritan-playout', '-f', 'Modelfile'], check=True)
lst = subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout
assert 'samaritan-playout' in lst, 'alias not created:\n' + lst
print(lst)
print('warming (first load ~30-60s)...')
body = json.dumps({'model': 'samaritan-playout',
    'messages': [{'role': 'user', 'content': 'What is 2+2? Reply with just the number.'}],
    'stream': False, 'options': {'num_predict': 64}}).encode()
req = urllib.request.Request('http://127.0.0.1:11434/api/chat', data=body,
    headers={'Content-Type': 'application/json'})
r = json.loads(urllib.request.urlopen(req, timeout=600).read())
print('[OK] model generates:', r['message']['content'][:80].replace(chr(10), ' '))

## 4. Tunnel + end-to-end verification

Publishes port 11434 (with `--http-host-header localhost:11434`, or Ollama 403s
the public hostname), then **hits the public URL through the tunnel and confirms
`/v1/models` lists the model and a real chat completes** — retrying while the
edge propagates. It prints the paste-ready lines *only* if that passes; on
failure it says whether Ollama or the tunnel is the problem.

In [ ]:
import subprocess, re, time, urllib.request, json
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
try:
    cf.terminate()
except Exception:
    pass
cf = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:11434',
                       '--http-host-header', 'localhost:11434'],
                      stdout=open('cf.log', 'w'), stderr=subprocess.STDOUT)
public = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('cf.log').read())
    if m:
        public = m.group(0); break
assert public, 'no tunnel URL — cf.log:\n' + open('cf.log').read()[-1000:]

def _get(url, timeout=60):
    req = urllib.request.Request(url, headers={'Authorization': 'Bearer ollama'})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, r.read().decode()

ok = False
for attempt in range(8):
    try:
        s, b = _get(public + '/v1/models')
        if s == 200 and 'samaritan-playout' in b:
            ok = True; break
        print(f'  try {attempt+1}: /v1/models -> {s} {b[:100]}')
    except Exception as e:
        print(f'  try {attempt+1}: {e}')
    time.sleep(5)

if not ok:
    print('\n[FAIL] public URL not ready. Diagnosing:')
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=5)
        print('  ollama LOCAL: UP  -> so it is the tunnel. cf.log tail:')
        print(open('cf.log').read()[-800:])
    except Exception as e:
        print('  ollama LOCAL: DOWN ->', e, '\n  Re-run cell 2 (and cell 3).')
    raise SystemExit('not ready — see above')

# A real completion through the tunnel, to prove the whole path.
body = json.dumps({'model': 'samaritan-playout',
    'messages': [{'role': 'user', 'content': 'What is 6 times 7? Reply with just the number.'}],
    'max_tokens': 256}).encode()
req = urllib.request.Request(public + '/v1/chat/completions', data=body,
    headers={'Authorization': 'Bearer ollama', 'Content-Type': 'application/json'})
reply = json.loads(urllib.request.urlopen(req, timeout=300).read())
print('[OK] chat through tunnel:', reply['choices'][0]['message']['content'][-80:].replace(chr(10), ' '))

print('\n[READY] paste into your LOCAL terminal:\n')
print(f'  $env:SAMARITAN_URL = "{public}/v1"')
print(f'  $env:SAMARITAN_API_KEY = "ollama"')
print(f'  $env:SAMARITAN_MODEL = "samaritan-playout:latest"')
print(f'\n  export SAMARITAN_URL="{public}/v1"   # bash')

## On your laptop

Paste the three lines cell 4 printed, then run a hard eval (RESUME makes it
crash-safe — a Colab drop costs only the unfinished items):

```powershell
$env:SAMARITAN_URL = "https://<from-cell-4>.trycloudflare.com/v1"
$env:SAMARITAN_API_KEY = "ollama"; $env:SAMARITAN_MODEL = "samaritan-playout:latest"
$env:MAX_TOKENS = "32768"; $env:LIMIT = "30"
$env:RESUME = "$env:USERPROFILE\models\reasoning\aime25.progress.jsonl"
$env:DATASET = "$env:USERPROFILE\models\reasoning\aime25.jsonl"
cargo run -p samaritan-run --example reason_eval
```

`MAX_TOKENS` is the generation budget (~32k ≈ 10-18 min/question); the 128k
`num_ctx` from cell 3 is the *window*, so the trace never gets silently trimmed
mid-derivation. A problem that still can't finish inside 32k un-shifted tokens
is a genuine capability miss, not a budget one — raise `MAX_TOKENS` (the window
holds up to ~120k) only to probe that ceiling.

**If Colab drops mid-run:** come back here, **Run all** (new tunnel URL), update
`$env:SAMARITAN_URL` to it, and rerun the *same* command — RESUME skips what's
done. Accuracy is reported over items actually answered; tunnel failures are
counted separately, not as wrong.

## Health check — run this anytime to see what's alive

One cell that reports the GPU, the Ollama daemon, the model, the tunnel, and the
public URL — and reprints the current `SAMARITAN_URL`. Run it if the laptop
starts erroring: it tells you whether to re-run the Serve cell, the Model cell,
or the Tunnel cell.

In [ ]:
import os, re, subprocess, urllib.request
def _get(url, timeout=15):
    req = urllib.request.Request(url, headers={'Authorization': 'Bearer ollama'})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, r.read().decode()
print('GPU :', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.total',
    '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
try:
    _get('http://127.0.0.1:11434/api/version'); print('daemon :', 'UP')
except Exception as e:
    print('daemon :', 'DOWN ->', e, '(re-run cell 2)')
try:
    _, b = _get('http://127.0.0.1:11434/api/tags')
    print('model  :', 'present' if 'samaritan-playout' in b else 'MISSING (re-run cell 3)')
except Exception as e:
    print('model  :', 'unknown ->', e)
url = None
if os.path.exists('cf.log'):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('cf.log').read())
    url = m.group(0) if m else None
try:
    alive = cf.poll() is None
except Exception:
    alive = False
print('tunnel :', ('alive' if alive else 'DOWN'), '| url:', url)
if url:
    try:
        s, b = _get(url + '/v1/models', 60)
        good = s == 200 and 'samaritan-playout' in b
        print('public :', s, 'OK' if good else '(model missing)')
        if good:
            print(f'\n  $env:SAMARITAN_URL = "{url}/v1"')
    except Exception as e:
        print('public :', 'FAILED ->', e, '(re-run cell 4 for a fresh URL)')

## Optional: self-training fine-tune here too

Independent of the served 27B: generate the verified set locally
(`selftrain_export`), upload it, and QDoRA-train the **local 4B** on this A100.

In [ ]:
# from google.colab import files; files.upload()   # reasoning-selftrain.jsonl + qdora_deviant.py + requirements.txt
# !pip -q install -r requirements.txt
# !python qdora_deviant.py reasoning-selftrain.jsonl --base-model Qwen/Qwen3-4B-Thinking-2507 --allow-small
print('uncomment to train; see training/README.md')

## Done?

There is deliberately **no shutdown cell** — in a Run-all it would fire last and
tear down the tunnel and Ollama the instant cell 4 finished verifying them, so
the laptop would hit 530. Stop it yourself when finished: **Runtime → Disconnect
and delete runtime** frees the A100 (stopping the tunnel alone doesn't).